In [1]:

import requests
import pandas as pd
import random
import time

BASE = "https://data.seattle.gov/resource/tmmm-ytt6.json"
YEARS = range(2006, 2026)
ROWS_PER_YEAR = 100_000
BLOCK_SIZE = 5_000

session = requests.Session()
rng = random.Random(42)
all_data = []

for year in YEARS:
    print(f"Downloading {year}...")

    where = f"checkoutyear={year}"

    count_response = session.get(
        BASE,
        params={
            "$select": "count(*) as total",
            "$where": where
        },
        timeout=120
    )
    count_response.raise_for_status()
    total = int(count_response.json()[0]["total"])

    target = min(ROWS_PER_YEAR, total)
    block_count = (target + BLOCK_SIZE - 1) // BLOCK_SIZE

    # Divide the year into sections and sample
    # one block from each section.
    for i in range(block_count):
        section_start = i * total // block_count
        section_end = (i + 1) * total // block_count

        size = min(BLOCK_SIZE, target - i * BLOCK_SIZE)
        max_offset = max(
            section_start,
            section_end - size
        )
        offset = rng.randint(section_start, max_offset)

        params = {
            "$where": where,
            "$order": ":id",
            "$limit": size,
            "$offset": offset
        }

        response = session.get(
            BASE, params=params, timeout=180
        )
        response.raise_for_status()
        all_data.extend(response.json())
        time.sleep(0.3)

    print(f"Finished {year}")

df = pd.DataFrame(all_data)
df.to_csv(
    "seattle_book_checkouts_sample.csv",
    index=False
)

print(f"Saved {len(df):,} rows")


Finished 2006


KeyboardInterrupt: 

In [2]:

import pandas as pd
from pathlib import Path

output_dir = Path("book_data")
output_dir.mkdir(exist_ok=True)

# Recover records from the interrupted script
if "all_data" in globals() and len(all_data) > 0:
    df_recovered = pd.DataFrame(all_data)

    print(f"Recovered {len(df_recovered):,} rows")
    print("Years in recovered data:")
    print(df_recovered["checkoutyear"].value_counts().sort_index())

    # Save each year separately
    for year, group in df_recovered.groupby("checkoutyear"):
        filename = output_dir / f"checkouts_{year}.csv"
        group.to_csv(filename, index=False)
        print(f"Saved {filename}: {len(group):,} rows")
else:
    print("No data found to recover.")


Recovered 140,000 rows
Years in recovered data:
checkoutyear
2006    100000
2007     40000
Name: count, dtype: int64
Saved book_data/checkouts_2006.csv: 100,000 rows
Saved book_data/checkouts_2007.csv: 40,000 rows


In [3]:

import requests
import pandas as pd
import random
import time
import json
from pathlib import Path

BASE = "https://data.seattle.gov/resource/tmmm-ytt6.json"

YEARS = range(2006, 2026)
ROWS_PER_YEAR = 100_000
BLOCK_SIZE = 5_000

output_dir = Path("book_data/resumable")
output_dir.mkdir(parents=True, exist_ok=True)

session = requests.Session()
rng = random.Random(42)

for year in YEARS:
    progress_file = output_dir / f"progress_{year}.json"
    data_file = output_dir / f"checkouts_{year}.csv"

    where = f"checkoutyear={year}"

    # Retrieve or initialize progress
    if progress_file.exists():
        progress = json.loads(progress_file.read_text())
    else:
        response = session.get(
            BASE,
            params={
                "$select": "count(*) as total",
                "$where": where
            },
            timeout=120
        )
        response.raise_for_status()
        total = int(response.json()[0]["total"])

        target = min(ROWS_PER_YEAR, total)
        block_count = (
            target + BLOCK_SIZE - 1
        ) // BLOCK_SIZE

        offsets = []

        for i in range(block_count):
            start = i * total // block_count
            end = (i + 1) * total // block_count
            size = min(
                BLOCK_SIZE,
                target - i * BLOCK_SIZE
            )
            max_offset = max(start, end - size)

            offsets.append({
                "offset": rng.randint(start, max_offset),
                "size": size
            })

        progress = {
            "year": year,
            "total": total,
            "blocks": offsets,
            "completed": 0
        }
        progress_file.write_text(
            json.dumps(progress, indent=2)
        )

    blocks = progress["blocks"]
    completed = progress["completed"]

    print(
        f"\nYear {year}: "
        f"{completed}/{len(blocks)} blocks complete"
    )

    for i in range(completed, len(blocks)):
        block = blocks[i]

        params = {
            "$where": where,
            "$order": ":id",
            "$limit": block["size"],
            "$offset": block["offset"]
        }

        for attempt in range(3):
            try:
                response = session.get(
                    BASE, params=params, timeout=120
                )
                response.raise_for_status()
                records = response.json()
                break
            except requests.RequestException:
                if attempt == 2:
                    raise
                time.sleep(5 * (attempt + 1))

        if len(records) != block["size"]:
            raise RuntimeError(
                f"Unexpected block size for {year}, "
                f"block {i}: {len(records)} rows"
            )

        df_block = pd.DataFrame(records)

        # Append rows to yearly CSV
        df_block.to_csv(
            data_file,
            mode="a",
            header=not data_file.exists(),
            index=False
        )

        # Mark block complete after saving
        progress["completed"] = i + 1
        progress_file.write_text(
            json.dumps(progress, indent=2)
        )

        print(
            f"  Block {i+1}/{len(blocks)} saved",
            flush=True
        )

        time.sleep(0.3)

    print(f"Finished {year}!")

print("\nAll downloads complete!")



Year 2006: 0/20 blocks complete
  Block 1/20 saved
  Block 2/20 saved
  Block 3/20 saved
  Block 4/20 saved
  Block 5/20 saved
  Block 6/20 saved
  Block 7/20 saved
  Block 8/20 saved
  Block 9/20 saved
  Block 10/20 saved
  Block 11/20 saved
  Block 12/20 saved
  Block 13/20 saved
  Block 14/20 saved
  Block 15/20 saved
  Block 16/20 saved
  Block 17/20 saved
  Block 18/20 saved
  Block 19/20 saved
  Block 20/20 saved
Finished 2006!

Year 2007: 0/20 blocks complete
  Block 1/20 saved
  Block 2/20 saved
  Block 3/20 saved
  Block 4/20 saved
  Block 5/20 saved
  Block 6/20 saved
  Block 7/20 saved
  Block 8/20 saved
  Block 9/20 saved
  Block 10/20 saved
  Block 11/20 saved
  Block 12/20 saved
  Block 13/20 saved
  Block 14/20 saved
  Block 15/20 saved
  Block 16/20 saved
  Block 17/20 saved
  Block 18/20 saved
  Block 19/20 saved
  Block 20/20 saved
Finished 2007!

Year 2008: 0/20 blocks complete
  Block 1/20 saved
  Block 2/20 saved
  Block 3/20 saved
  Block 4/20 saved
  Block 5/20 